In [0]:
%sql
-- OWID · Paso 1 (estructura): filas de cada tabla. Dos tablas bronze separadas (D12).
SELECT 'owid_cantril_ladder' AS tabla, COUNT(*) AS filas FROM happiness.bronze.owid_cantril_ladder
UNION ALL
SELECT 'owid_gdp_happiness', COUNT(*) FROM happiness.bronze.owid_gdp_happiness;

In [0]:
%sql
-- OWID · Paso 1 (estructura): tipos de columna de cantril_ladder. Todo STRING salvo _ingest_ts (D10/D15).
DESCRIBE TABLE happiness.bronze.owid_cantril_ladder;

In [0]:
%sql
-- OWID · Paso 1 (estructura): tipos de columna de cantril_ladder. Todo STRING salvo _ingest_ts (D10/D15).
DESCRIBE TABLE happiness.bronze.owid_cantril_ladder;

-- OWID · Paso 1 (estructura): tipos de columna de gdp_happiness.
DESCRIBE TABLE happiness.bronze.owid_gdp_happiness;

In [0]:
%sql
-- OWID · Paso 1 (estructura): muestra de cada tabla. 'Code' es el ISO3 de OWID (vacío para agregados).
SELECT * FROM happiness.bronze.owid_cantril_ladder LIMIT 5;

SELECT * FROM happiness.bronze.owid_gdp_happiness LIMIT 5;

In [0]:
%sql
-- OWID cantril_ladder · Paso 2 (nulos): claves y puntaje. Se separan NULL real, vacío y texto 'null'.
SELECT COUNT(*) AS filas,
       COUNT_IF(Entity IS NULL OR Entity IN ('', 'null'))  AS entity_faltante,
       COUNT_IF(Code IS NULL)                              AS code_null_real,
       COUNT_IF(Code IN ('', 'null'))                      AS code_vacio_o_texto,
       COUNT_IF(Year IS NULL OR Year IN ('', 'null'))      AS anio_faltante,
       COUNT_IF(`Self-reported life satisfaction` IS NULL OR `Self-reported life satisfaction` IN ('', 'null')) AS puntaje_faltante,
       COUNT_IF(_rescued_data IS NOT NULL)                 AS filas_rescatadas
FROM happiness.bronze.owid_cantril_ladder;

In [0]:
%sql
-- OWID gdp_happiness · Paso 2 (nulos): claves, puntaje, PIB y región.
SELECT COUNT(*) AS filas,
       COUNT_IF(Entity IS NULL OR Entity IN ('', 'null'))  AS entity_faltante,
       COUNT_IF(Code IS NULL)                              AS code_null_real,
       COUNT_IF(Code IN ('', 'null'))                      AS code_vacio_o_texto,
       COUNT_IF(Year IS NULL OR Year IN ('', 'null'))      AS anio_faltante,
       COUNT_IF(`Life satisfaction` IS NULL OR `Life satisfaction` IN ('', 'null'))   AS puntaje_faltante,
       COUNT_IF(`GDP per capita` IS NULL OR `GDP per capita` IN ('', 'null'))         AS pib_faltante,
       COUNT_IF(`World region according to OWID` IS NULL OR `World region according to OWID` IN ('', 'null')) AS region_faltante,
       COUNT_IF(_rescued_data IS NOT NULL)                 AS filas_rescatadas
FROM happiness.bronze.owid_gdp_happiness;

In [0]:
%sql
-- OWID · Paso 3 (cardinalidad): entidades, códigos y años distintos en cada tabla.
-- Si hay más entidades que códigos es porque hay agregados sin código (o nombres repetidos).
SELECT 'cantril_ladder' AS tabla, COUNT(DISTINCT Entity) AS entidades, COUNT(DISTINCT Code) AS codigos,
       MIN(TRY_CAST(Year AS INT)) AS primer_anio, MAX(TRY_CAST(Year AS INT)) AS ultimo_anio
FROM happiness.bronze.owid_cantril_ladder
UNION ALL
SELECT 'gdp_happiness', COUNT(DISTINCT Entity), COUNT(DISTINCT Code),
       MIN(TRY_CAST(Year AS INT)), MAX(TRY_CAST(Year AS INT))
FROM happiness.bronze.owid_gdp_happiness;

In [0]:
%sql
-- OWID · Paso 5 (duplicados): clave natural (Entity, Year) en cada tabla. Esperado: 0 repetidas.
WITH c AS (
  SELECT ROW_NUMBER() OVER (PARTITION BY Entity, Year ORDER BY _ingest_ts) AS rn FROM happiness.bronze.owid_cantril_ladder
), g AS (
  SELECT ROW_NUMBER() OVER (PARTITION BY Entity, Year ORDER BY _ingest_ts) AS rn FROM happiness.bronze.owid_gdp_happiness
)
SELECT 'cantril_ladder' AS tabla, COUNT(*) AS filas, COUNT_IF(rn > 1) AS claves_duplicadas FROM c
UNION ALL
SELECT 'gdp_happiness', COUNT(*), COUNT_IF(rn > 1) FROM g;

In [0]:
%sql
-- OWID · Paso 4 (rangos): puntaje de satisfacción (escala 0-10) y PIB per cápita.
-- 'no_numericos' cuenta valores no vacíos que no se pueden convertir (problema para el CAST de Silver).
SELECT 'cantril_ladder' AS tabla,
       MIN(TRY_CAST(`Self-reported life satisfaction` AS DOUBLE)) AS puntaje_min,
       MAX(TRY_CAST(`Self-reported life satisfaction` AS DOUBLE)) AS puntaje_max,
       NULL AS pib_min, NULL AS pib_max,
       COUNT_IF(`Self-reported life satisfaction` NOT IN ('', 'null') AND TRY_CAST(`Self-reported life satisfaction` AS DOUBLE) IS NULL) AS no_numericos
FROM happiness.bronze.owid_cantril_ladder
UNION ALL
SELECT 'gdp_happiness',
       MIN(TRY_CAST(`Life satisfaction` AS DOUBLE)), MAX(TRY_CAST(`Life satisfaction` AS DOUBLE)),
       MIN(TRY_CAST(`GDP per capita` AS DOUBLE)),    MAX(TRY_CAST(`GDP per capita` AS DOUBLE)),
       COUNT_IF((`Life satisfaction` NOT IN ('', 'null') AND TRY_CAST(`Life satisfaction` AS DOUBLE) IS NULL)
             OR (`GDP per capita` NOT IN ('', 'null') AND TRY_CAST(`GDP per capita` AS DOUBLE) IS NULL))
FROM happiness.bronze.owid_gdp_happiness;

In [0]:
%sql
-- OWID · Paso 6 (temporal): filas por año y cuántas traen puntaje, para cada tabla.
-- Sirve para ver desde qué año hay puntaje y si cantril_ladder y gdp_happiness cubren los mismos años.
SELECT 'cantril_ladder' AS tabla, TRY_CAST(Year AS INT) AS anio, COUNT(*) AS filas,
       COUNT_IF(`Self-reported life satisfaction` IS NOT NULL AND `Self-reported life satisfaction` NOT IN ('', 'null')) AS con_puntaje
FROM happiness.bronze.owid_cantril_ladder GROUP BY ALL
UNION ALL
SELECT 'gdp_happiness', TRY_CAST(Year AS INT), COUNT(*),
       COUNT_IF(`Life satisfaction` IS NOT NULL AND `Life satisfaction` NOT IN ('', 'null'))
FROM happiness.bronze.owid_gdp_happiness GROUP BY ALL
ORDER BY tabla, anio;

In [0]:
%sql
-- OWID · Para dim_pais_base: entidades cuyo Code no existe en Banco Mundial (o no tiene Code).
-- Incluye agregados (continentes, 'World') y casos con código propio de OWID (OWID_KOS, OWID_CYN...).
-- 'Code' en OWID es ISO3; lo que no coincida acá hay que resolver a mano o excluir.
WITH owid AS (
  SELECT DISTINCT Entity, Code FROM happiness.bronze.owid_cantril_ladder
  UNION
  SELECT DISTINCT Entity, Code FROM happiness.bronze.owid_gdp_happiness
),
bm AS (SELECT DISTINCT `Country Code` AS codigo FROM happiness.bronze.banco_mundial)
SELECT o.Entity, o.Code
FROM owid o LEFT JOIN bm ON o.Code = bm.codigo
WHERE bm.codigo IS NULL
ORDER BY o.Code, o.Entity;

## Conclusiones OWID
- cantril_ladder: 2.270 filas, 178 países, 2011-2025 (sin 2013). Sin nulos ni duplicados. Puntaje 1,364-7,856, igual al de WHR.
- gdp_happiness: 7.594 filas, 225 entidades, 1990-2025. Puntaje solo en 2011+ (2.270 valores, = cantril_ladder). PIB 510,8-174.569,5, igual a NY.GDP.PCAP.PP.KD de Banco Mundial. 149 PIB y 565 regiones vacíos.
- Code = ISO3, salvo 23 entidades que no están en Banco Mundial: 19 agregados (OWID_*, WB_*), Kosovo (OWID_KOS → XKX), Taiwán (TWN) y dos sin ISO3 (OWID_CYN Northern Cyprus, OWID_SML Somaliland).
- OWID sirve de cruce contra WHR y Banco Mundial; no aporta variables nuevas.
- Detalle en eda/hallazgos_paises.md.
- Puntaje OWID vs WHR: de 2.270 filas, 1.948 cruzan por (nombre, año) y 0 difieren (diferencia máxima 0,0005 por redondeo). Mismo dato. 322 filas sin cruce por nombres distintos.

In [0]:
%sql
-- OWID vs WHR · ¿El puntaje de OWID (cantril_ladder) es el mismo que el de WHR?
-- Se cruza por (nombre de país, año). Los ~20 países con nombre distinto quedan sin cruzar acá:
-- se miden aparte (columna sin_cruce) y se resuelven con la tabla de equivalencias.
WITH owid AS (
  SELECT Entity AS pais, TRY_CAST(Year AS INT) AS anio,
         TRY_CAST(`Self-reported life satisfaction` AS DOUBLE) AS puntaje_owid
  FROM happiness.bronze.owid_cantril_ladder
),
whr AS (
  SELECT `Country name` AS pais, CAST(TRY_CAST(`Year` AS DOUBLE) AS INT) AS anio,
         TRY_CAST(`Life evaluation (3-year average)` AS DOUBLE) AS puntaje_whr
  FROM happiness.bronze.whr
)
SELECT COUNT(*)                                         AS filas_owid,
       COUNT(w.puntaje_whr)                             AS cruzadas_por_nombre,
       COUNT_IF(w.puntaje_whr IS NULL)                  AS sin_cruce,
       COUNT_IF(ABS(o.puntaje_owid - w.puntaje_whr) > 0.001) AS puntaje_distinto,
       MAX(ABS(o.puntaje_owid - w.puntaje_whr))         AS diferencia_maxima
FROM owid o LEFT JOIN whr w ON o.pais = w.pais AND o.anio = w.anio;

Probado:

El puntaje de cantril_ladder es idéntico al de WHR en las 1.948 filas que cruzamos por nombre (0 diferencias).
En gdp_happiness, el puntaje tiene el mismo número de valores (2.270) y el mismo rango que cantril_ladder.

Todavía sin probar (solo coinciden los rangos, no comparé fila por fila):

Que el puntaje de gdp_happiness sea igual al de cantril_ladder país por país.
Que el "GDP per capita" de OWID sea igual a NY.GDP.PCAP.PP.KD de Banco Mundial. Coinciden el mínimo y el máximo exactos, lo cual es una señal muy fuerte, pero no una prueba.
Que OWID no tenga ningún país-año que WHR no tenga. cantril_ladder trae 154 filas más que WHR (2.270 contra 2.116), y las 322 filas sin cruce por nombre mezclan nombres distintos con posibles países extra. Esa es la pregunta clave: si OWID tiene países-año que WHR no tiene, no sobra del todo.

Qué tablas serían prescindibles si todo se confirma:

owid_gdp_happiness: casi seguro. Su puntaje repite a cantril_ladder, su PIB repite a Banco Mundial, y lo único propio es World region according to OWID. Esa región se puede sacar de Banco Mundial o WGI.
owid_cantril_ladder: depende de si tiene países-año que WHR no tiene. Si no tiene ninguno, sobra como fuente. Aun así podría servir como chequeo de calidad cruzado en S8 (comparar WHR contra una segunda copia del mismo dato).

In [0]:
%sql
-- OWID · ¿El puntaje de gdp_happiness es igual, país por país, al de cantril_ladder?
-- Se cruza por (Code, Year). 'sin_par' = puntaje en gdp_happiness que cantril_ladder no tiene.
SELECT COUNT(*)                                        AS filas_con_puntaje,
       COUNT_IF(c.Code IS NULL)                        AS sin_par_en_cantril,
       COUNT_IF(ABS(TRY_CAST(g.`Life satisfaction` AS DOUBLE)
                  - TRY_CAST(c.`Self-reported life satisfaction` AS DOUBLE)) > 0.001) AS puntaje_distinto
FROM happiness.bronze.owid_gdp_happiness g
LEFT JOIN happiness.bronze.owid_cantril_ladder c ON g.Code = c.Code AND g.Year = c.Year
WHERE g.`Life satisfaction` IS NOT NULL;

In [0]:
%sql
-- OWID · ¿El PIB per cápita de OWID es igual al de Banco Mundial (NY.GDP.PCAP.PP.KD) país por país?
-- Se despivota solo ese indicador de Banco Mundial (ancho → largo, en memoria, sin crear tabla)
-- y se cruza por (Code, Year). Tolerancia 0,01 porque OWID redondea a 4 decimales.
WITH bm AS (
  SELECT `Country Code` AS codigo, anio, TRY_CAST(valor AS DOUBLE) AS pib_bm
  FROM (SELECT * FROM happiness.bronze.banco_mundial WHERE `Indicator Code` = 'NY.GDP.PCAP.PP.KD')
  UNPIVOT (valor FOR anio IN (
    `1990`, `1991`, `1992`, `1993`, `1994`, `1995`, `1996`, `1997`, `1998`, `1999`, `2000`, `2001`, `2002`, `2003`, `2004`, `2005`, `2006`, `2007`, `2008`, `2009`, `2010`, `2011`, `2012`, `2013`, `2014`, `2015`, `2016`, `2017`, `2018`, `2019`, `2020`, `2021`, `2022`, `2023`, `2024`, `2025`
  ))
)
SELECT COUNT(*)                                                       AS filas_owid_con_pib,
       COUNT(bm.pib_bm)                                               AS cruzan_con_banco_mundial,
       COUNT_IF(bm.pib_bm IS NOT NULL
                AND ABS(TRY_CAST(g.`GDP per capita` AS DOUBLE) - bm.pib_bm) > 0.01) AS pib_distinto,
       COUNT_IF(bm.pib_bm IS NULL)                                    AS sin_dato_en_bm
FROM happiness.bronze.owid_gdp_happiness g
LEFT JOIN bm ON g.Code = bm.codigo AND g.Year = bm.anio
WHERE g.`GDP per capita` IS NOT NULL;

In [0]:
%sql
-- OWID vs WHR · Nombres de país que no coinciden entre las dos fuentes (cantril_ladder vs whr).
-- 'solo_owid' trae el Code (ISO3) de OWID: sirve para confirmar a qué país de WHR corresponde.
-- 'solo_whr' son los nombres de WHR que OWID escribe distinto o no trae.
WITH owid AS (SELECT DISTINCT Entity AS nombre, Code AS codigo FROM happiness.bronze.owid_cantril_ladder),
     whr  AS (SELECT DISTINCT `Country name` AS nombre FROM happiness.bronze.whr)
SELECT 'solo_owid' AS donde, o.nombre, o.codigo
FROM owid o LEFT ANTI JOIN whr w ON o.nombre = w.nombre
UNION ALL
SELECT 'solo_whr', w.nombre, NULL
FROM whr w LEFT ANTI JOIN owid o ON w.nombre = o.nombre
ORDER BY donde, nombre;

## Decisión sobre OWID
- gdp_happiness: puntaje idéntico a cantril_ladder (2.270/2.270) y PIB idéntico a Banco Mundial (6.958 de 6.959 cruzadas). Sin variables propias salvo la región OWID.
- cantril_ladder: idéntica a WHR (1.948/1.948), sin países reales extra (solo 11 agregados).
- Uso propuesto: puente nombre→ISO3 de WHR (dim_pais_base) y chequeo cruzado de calidad (S8). No se pasa a Silver como fuente de datos.